# Lab 6

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giswqs/geog-414/blob/master/book/labs/lab_06.ipynb)

## Submission requirements

1. Upload a screenshot of your map for each question.
2. Provide a link to your notebook on Colab. See instructions [here](https://geog-414.gishub.org/book/labs/instructions.html).

## Question 1

Create a map to visualize [NOAA GFS Temperature Data](https://developers.google.com/earth-engine/datasets/catalog/NOAA_GFS0P25) and add a color bar and [NOAA logo](https://i.imgur.com/spILFEi.png) to the map.

In [1]:
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='lab3-515123')
print('OK')

OK


In [ ]:
import ee
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
import numpy as np

# بارگذاری آخرین تصویر NOAA GFS 0.25-degree و تبدیل کلوین به سانتی‌گراد
gfs_temp_c = (
    ee.ImageCollection('NOAA/GFS0P25')
    .sort('system:time_start', False)
    .first()
    .select('temperature_2m_above_ground')
    .subtract(273.15)
)

# استفاده از مقدار عددی معتبر (-9999) به جای np.nan و مقیاس ~1 درجه
region = ee.Geometry.BBox(-180, -85, 180, 85)
arr = (
    gfs_temp_c.sampleRectangle(region=region, defaultValue=-9999)
    .get('temperature_2m_above_ground')
    .getInfo()
)

temp_grid = np.array(arr, dtype=float)
temp_grid = np.where(temp_grid <= -9000, np.nan, temp_grid)

palette = [
    '#313695',
    '#4575b4',
    '#74add1',
    '#abd9e9',
    '#fee090',
    '#fdae61',
    '#f46d43',
    '#d73027',
    '#a50026',
]
cmap = LinearSegmentedColormap.from_list('noaa_temp', palette, N=256)

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=150)
im = ax.imshow(
    temp_grid,
    cmap=cmap,
    vmin=-35,
    vmax=40,
    extent=[-180, 180, -85, 85],
    origin='upper',
)
ax.set_title('NOAA GFS 2m Temperature (°C)', fontsize=13, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.set_label('Temperature (°C)')

ax.text(
    0.01,
    0.02,
    'NOAA | Made by Mahdieh Zakizadeh',
    transform=ax.transAxes,
    fontsize=10,
    color='black',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8),
)

plt.tight_layout()
plt.show()

![](https://i.imgur.com/bBoRswR.png)

## Question 2

**Linked Maps:** Create a 2*2 linked map to visualize the Landsat imagery (`ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003')`) with different band combinations.

In [ ]:
from io import BytesIO
import ee
import matplotlib.pyplot as plt
from PIL import Image
import requests

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

landsat = ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003')

# دلتای نیل
region = ee.Geometry.Rectangle([29.5, 29.8, 32.8, 31.8])

band_sets = [
    ('True Color (B3, B2, B1)', ['B3', 'B2', 'B1'], 1.3),
    ('False Color Vegetation (B4, B3, B2)', ['B4', 'B3', 'B2'], 1.2),
    ('Agriculture / SWIR (B5, B4, B3)', ['B5', 'B4', 'B3'], 1.2),
    ('Urban / Geology (B7, B5, B3)', ['B7', 'B5', 'B3'], 1.2),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=150)
axes = axes.flatten()

for ax, (title, bands, gamma) in zip(axes, band_sets):
  # محاسبه خودکار min و max بر اساس صدک 2 و 98 برای جلوگیری از سفیدی یا تیرگی بیش از حد
  stats = (
      landsat.select(bands)
      .reduceRegion(
          reducer=ee.Reducer.percentile([2, 98]),
          geometry=region,
          scale=120,
          maxPixels=1e7,
          bestEffort=True,
      )
      .getInfo()
  )

  mins = [stats.get(f'{b}_p2', 0.0) for b in bands]
  maxs = [stats.get(f'{b}_p98', 0.4) for b in bands]

  vis_params = {
      'bands': bands,
      'min': mins,
      'max': maxs,
      'gamma': gamma,
      'region': region,
      'dimensions': 512,
  }

  url = landsat.getThumbURL(vis_params)
  resp = requests.get(url, timeout=30)
  resp.raise_for_status()

  img = Image.open(BytesIO(resp.content)).convert('RGB')
  ax.imshow(img)
  ax.set_title(title, fontsize=11, fontweight='bold')
  ax.axis('off')

plt.tight_layout()
plt.show()

![](https://i.imgur.com/Xp15uOo.png)

## Question 3

**Timeseries Inspector:** Create a map with timeseries inspector to visualize [USDA NASS Cropland Data Layers](https://developers.google.com/earth-engine/datasets/catalog/USDA_NASS_CDL) from 2010 to 2022. Add your name and [USDA logo](https://i.imgur.com/tzH2dNr.png) to the map.

In [ ]:
from io import BytesIO
import os
import ee
import geemap
from ipyleaflet import TileLayer, WidgetControl
import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image
import requests

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

cdl_col = (
    ee.ImageCollection('USDA/NASS/CDL')
    .filterDate('2010-01-01', '2022-12-31')
    .select('cropland')
)

vis_params = {
    'bands': ['cropland'],
    'min': 1,
    'max': 240,
    'palette': [
        '#ffd300',
        '#ff2626',
        '#267300',
        '#a87000',
        '#70a800',
        '#e9ffbe',
        '#d3ffbe',
        '#00a884',
    ],
}

esri_clean = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)


def clean_map_osm(target_map):
  if target_map is None or not hasattr(target_map, 'layers'):
    return
  kept = []
  for lyr in target_map.layers:
    name = str(getattr(lyr, 'name', '')).lower()
    url = str(getattr(lyr, 'url', '')).lower()
    if (
        'openstreetmap' in name
        or name == 'osm'
        or 'openstreetmap' in url
        or 'osm.org' in url
    ):
      continue
    kept.append(lyr)
  has_esri = any(
      'arcgisonline' in str(getattr(lyr, 'url', '')).lower() for lyr in kept
  )
  if not has_esri:
    kept.insert(0, esri_clean)
  target_map.layers = tuple(kept)


# ۱. ساخت نقشه و فعال‌سازی ts_inspector
m = geemap.Map(center=[-96.5, 41.0], zoom=6, height='480px')
m.ts_inspector(
    cdl_col,
    vis_params=vis_params,
    date_format='YYYY',
)

# حذف کامل لایه‌های OSM بعد از ساخت ts_inspector
clean_map_osm(m)
clean_map_osm(getattr(m, 'left_map', None))
clean_map_osm(getattr(m, 'right_map', None))

# ۲. افزودن نام شما و نشان USDA
m.add_text(
    'USDA NASS CDL (2010–2022) | Mahdieh Zakizadeh',
    fontsize=11,
    position='bottomleft',
)

usda_badge = widgets.HTML(
    '<div style="background: rgba(255,255,255,0.95); padding: 6px 10px; '
    'border-radius: 6px; border: 2px solid #2e6b3e; color: #1b4326; '
    'font-weight: bold; font-family: sans-serif;">USDA</div>'
)
m.add_control(WidgetControl(widget=usda_badge, position='bottomright'))

display(m)

# ۳. نمای مستقیم از لایه‌های 2010 و 2022 برای اطمینان از نمایش رنگی در Colab
region = ee.Geometry.Rectangle([-98.0, 40.2, -95.0, 41.8])
img_2010 = cdl_col.filterDate('2010-01-01', '2010-12-31').first()
img_2022 = cdl_col.filterDate('2022-01-01', '2022-12-31').first()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=140)
for ax, img, yr in zip(axes, [img_2010, img_2022], ['2010', '2022']):
  thumb_params = dict(vis_params)
  thumb_params['region'] = region
  thumb_params['dimensions'] = 512
  url = img.getThumbURL(thumb_params)
  resp = requests.get(url, timeout=30)
  resp.raise_for_status()
  pic = Image.open(BytesIO(resp.content)).convert('RGB')
  ax.imshow(pic)
  ax.set_title(f'USDA NASS CDL - {yr} (Cropland)', fontsize=11)
  ax.axis('off')

plt.tight_layout()
plt.show()

![](https://i.imgur.com/gW1XPhL.png)

## Question 4

**Time slider:** Create a map with the time slider to visualize [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR) for Knoxville, TN.

In [ ]:
from io import BytesIO
import os
import ee
import geemap
from ipyleaflet import TileLayer
import matplotlib.pyplot as plt
from PIL import Image
import requests

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

# محدوده Knoxville, TN
knoxville_pt = ee.Geometry.Point([-83.9207, 35.9606])
knoxville_region = knoxville_pt.buffer(12000).bounds()

# فیلتر Sentinel-2 Surface Reflectance (باندهای رنگ طبیعی B4, B3, B2)
s2_base = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(knoxville_region)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
    .select(['B4', 'B3', 'B2'])
)

# ساخت سری زمانی ماهانه با system:time_start معتبر برای add_time_slider
monthly_list = []
for month in range(1, 13):
  start = ee.Date.fromYMD(2023, month, 1)
  end = start.advance(1, 'month')
  comp = (
      s2_base.filterDate(start, end)
      .median()
      .multiply(0.0001)
      .clip(knoxville_region)
      .set('system:time_start', start.millis())
  )
  monthly_list.append(comp)

s2_ts = ee.ImageCollection(monthly_list)

vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.02,
    'max': 0.28,
    'gamma': 1.2,
}

# ۱. نقشه تعاملی همراه با Time Slider
m = geemap.Map(center=[35.9606, -83.9207], zoom=11, height='480px')

esri_layer = TileLayer(
    url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attribution='Esri',
    name='Esri World Street Map',
)
clean_layers = [
    lyr
    for lyr in m.layers
    if 'openstreetmap' not in str(getattr(lyr, 'name', '')).lower()
    and 'osm' not in str(getattr(lyr, 'name', '')).lower()
]
if not any(
    'arcgisonline' in str(getattr(lyr, 'url', '')).lower()
    for lyr in clean_layers
):
  clean_layers.insert(0, esri_layer)
m.layers = tuple(clean_layers)

m.add_time_slider(s2_ts, vis_params, time_interval=1)
display(m)

# ۲. خروجی قطعی دو فصل از Knoxville, TN برای اطمینان از وضوح تصویر در Colab
spring_img = (
    s2_base.filterDate('2023-04-01', '2023-05-31').median().multiply(0.0001)
)
summer_img = (
    s2_base.filterDate('2023-07-01', '2023-08-31').median().multiply(0.0001)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=140)
for ax, img, title in zip(
    axes,
    [spring_img, summer_img],
    ['Knoxville, TN - Spring 2023', 'Knoxville, TN - Summer 2023'],
):
  params = dict(vis_params)
  params['region'] = knoxville_region
  params['dimensions'] = 512
  resp = requests.get(img.getThumbURL(params), timeout=30)
  resp.raise_for_status()
  ax.imshow(Image.open(BytesIO(resp.content)).convert('RGB'))
  ax.set_title(title, fontsize=11)
  ax.axis('off')

plt.tight_layout()
plt.show()

![](https://i.imgur.com/rJe2rvP.png)

## Question 5

**Split-panel Map:** Use the following datasets to create a split-panel map for visualizing the ESA land cover data in the US. Add the ESA land cover legend to the map (Hint: the built-in legend for ESA land cover is `ESA_WorldCover`).

- [US Census States](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2018_States): `ee.FeatureCollection("TIGER/2018/States")`
- [ESA WorldCover 10m](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100): `ee.ImageCollection("ESA/WorldCover/v100")`
- Landsat: `LANDSAT/LE7_TOA_5YEAR/1999_2003`


Currently, the split-map control of ipyleaflet plotting backend has a bug ([source](https://github.com/jupyter-widgets/ipyleaflet/issues/1066)). Use the folium plotting backend instead.

In [2]:
import os
import ee
import folium
from folium.plugins import SideBySideLayers

os.environ.setdefault('GOOGLE_MAPS_API_KEY', '')

try:
  ee.Initialize(project='lab3-515123')
except Exception:
  ee.Authenticate(auth_mode='notebook')
  ee.Initialize(project='lab3-515123')

# ۱. مجموعه‌داده‌های Question 5
us_states = ee.FeatureCollection('TIGER/2018/States')
esa_lc = ee.ImageCollection('ESA/WorldCover/v100').first().clip(us_states)
landsat_99_03 = ee.Image('LANDSAT/LE7_TOA_5YEAR/1999_2003').clip(us_states)

# ۲. پارامترهای نمایش
esa_vis = {'bands': ['Map']}
landsat_vis = {'bands': ['B3', 'B2', 'B1'], 'min': 0.02, 'max': 0.35, 'gamma': 1.2}


def make_ee_layer(ee_object, vis_params, name, show=True):
  map_id = ee_object.getMapId(vis_params)
  return folium.TileLayer(
      tiles=map_id['tile_fetcher'].url_format,
      attr='Google Earth Engine',
      name=name,
      overlay=True,
      control=True,
      show=show,
  )


# ۳. ساخت نقشه Folium با لایه پایه تمیز Esri (بدون OSM و بدون geemap.Map)
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles=None)

folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri World Street Map',
    overlay=False,
    control=True,
).add_to(m)

left_layer = make_ee_layer(esa_lc, esa_vis, 'ESA WorldCover 10m', show=True)
right_layer = make_ee_layer(
    landsat_99_03, landsat_vis, 'Landsat 7 TOA (1999–2003)', show=True
)

left_layer.add_to(m)
right_layer.add_to(m)

# اسلایدر مقایسه چپ و راست
sbs = SideBySideLayers(layer_left=left_layer, layer_right=right_layer)
sbs.add_to(m)

# مرز ایالت‌های آمریکا
states_styled = us_states.style(
    color='#222222', fillColor='00000000', width=1
)
make_ee_layer(states_styled, {}, 'US States', show=True).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

# ۴. راهنمای رسمی طبقات ESA WorldCover + امضای شما
legend_and_title_html = """
<div style="position: fixed; bottom: 20px; right: 20px; z-index: 9999;
            background-color: rgba(255, 255, 255, 0.95); padding: 10px 12px;
            border-radius: 6px; border: 1px solid #666; font-size: 11px;
            font-family: sans-serif; box-shadow: 0 2px 6px rgba(0,0,0,0.25);
            max-width: 220px;">
    <div style="font-weight: bold; margin-bottom: 6px; font-size: 12px;">ESA WorldCover Legend</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#006400;margin-right:6px;"></span>Tree cover</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#ffbb22;margin-right:6px;"></span>Shrubland</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#ffff4c;margin-right:6px;"></span>Grassland</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#f096ff;margin-right:6px;"></span>Cropland</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#fa0000;margin-right:6px;"></span>Built-up</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#b4b4b4;margin-right:6px;"></span>Bare / sparse vegetation</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#f0f0f0;border:1px solid #ccc;margin-right:6px;"></span>Snow and ice</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#0064c8;margin-right:6px;"></span>Permanent water bodies</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#0096a0;margin-right:6px;"></span>Herbaceous wetland</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#00cf75;margin-right:6px;"></span>Mangroves</div>
    <div><span style="display:inline-block;width:12px;height:12px;background:#fae6a0;margin-right:6px;"></span>Moss and lichen</div>
</div>

<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999;
            background-color: rgba(255, 255, 255, 0.94); padding: 6px 12px;
            border-radius: 6px; border: 1px solid #444; font-size: 12px;
            font-family: sans-serif; box-shadow: 0 1px 4px rgba(0,0,0,0.25);">
    <b>ESA WorldCover &amp; Landsat (1999–2003)</b> | Mahdieh Zakizadeh
</div>
"""
m.get_root().html.add_child(folium.Element(legend_and_title_html))

m

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=0zYuUS-jr7mGUHhkPiUnErUvlzox2l7L7yuElychp-M&tc=6TjTKm2-B04b-Nq4TFe2tYth5obHiZX2IM-2vvArCSw&cc=O3P_iHcMI3pX7qp6M3R4CMHsjRDOOVjtIYEG3fCuAIk

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AUn6Hd-piXIMKMFAz6HspYHTcJqeD59tEPmJKJtjvgXKKbiUQO91AQBxPsc

Successfully saved authorization token.


![](https://i.imgur.com/y0GW6k8.png)